In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from shared.common import load_config

In [2]:
settings = load_config()
inp, out = Path('../input'), Path('../output')
n_folds = settings['field']['folds']
rng = np.random.default_rng(settings['seed'])

In [3]:
release = pd.read_csv(inp / 'releases.csv')
observed = pd.read_csv(inp / 'observations.csv')
wind = pd.read_csv(inp / 'wind_profiles.csv')

In [4]:
# intersection with the EPA evaluation set; keep observed sample windows
runs = release.dropna(subset=['wind_from_deg']).copy()
runs['stability'] = runs['stability'].astype(int)
runs['toward_deg'] = (runs['wind_from_deg'] + 180) % 360
profiles = wind.loc[wind['run'].isin(runs['run'])].dropna(subset=['speed_m_s'])
receptors = observed.loc[observed['run'].isin(runs['run'])].dropna(subset=['observed_mg_m3']).copy()
theta = np.deg2rad(receptors['bearing_deg'])
receptors['east_m'] = receptors['radius_m']*np.sin(theta)
receptors['north_m'] = receptors['radius_m']*np.cos(theta)

In [5]:
# block by archive met day; randomized tie order, greedy balance by run count
days = runs.groupby('met_day').size()
shuffled = days.reindex(rng.permutation(days.index.to_numpy())).sort_values(ascending=False, kind='stable')
fold_load = np.zeros(n_folds, dtype=int)
fold_by_day = {}
for day, count in shuffled.items():
    fold = int(np.argmin(fold_load))
    fold_by_day[day] = fold
    fold_load[fold] += count
runs['fold'] = runs['met_day'].map(fold_by_day)
receptors = receptors.merge(runs[['run', 'met_day', 'fold']], on='run', validate='many_to_one')

In [6]:
coverage = receptors.groupby(['fold', 'array'], as_index=False).agg(
    days=('met_day', 'nunique'), runs=('run', 'nunique'),
    measured=('obs_id', 'size'), observed_zero=('observed_mg_m3', lambda x: x.eq(0).sum()),
    corrected=('arc_factor', 'count'))
scope = release[['run', 'date', 'met_day']].merge(runs[['run', 'fold']], on='run', how='left')
scope['use'] = np.where(scope['fold'].notna(), 'EPA archive with met', 'outside EPA 44-run subset')
scope.to_csv(out / 'release_scope.csv', index=False)
coverage.to_csv(out / 'coverage.csv', index=False)
runs.to_csv(out / 'runs.csv', index=False)
profiles.to_csv(out / 'profiles.csv', index=False)
receptors.to_csv(out / 'receptors.csv', index=False)
display(coverage)

,fold,array,days,runs,measured,observed_zero,corrected
0,0,arc,4,9,778,0,740
1,0,tower,3,8,431,282,0
2,1,arc,4,9,1039,0,1039
3,1,tower,4,9,486,285,0
4,2,arc,4,9,991,0,991
5,2,tower,4,8,432,251,0
6,3,arc,4,9,951,0,951
7,3,tower,3,6,323,189,0
8,4,arc,4,8,988,0,988
9,4,tower,3,6,318,205,0
